# W4 Development: SARSA vs Q-Learning on CliffWalking

Craig Rudman | MSDS684 | TDD development notebook  
All implementation lives in `../src/`. Tests live in `../tests/`.  
Work through TODOs in sequence; flip each to DONE when complete.

---
## Phase 1: Infrastructure

In [ ]:
# TODO: Create stub modules in src/
#   schedules.py     -- EpsilonSchedule (base), ConstantSchedule, LinearDecaySchedule, ExponentialDecaySchedule
#   agents.py        -- TDAgent (base), SARSAAgent, QLearningAgent
#   environment.py   -- EnvironmentManager
#   runner.py        -- EpisodeRunner
#   experiment.py    -- ExperimentConfig, ExperimentResult (with save()), ExperimentSuite (with summarize(output_path=None))
#   visualization.py -- Visualizer: plot_learning_curves, plot_policy_arrows, plot_value_heatmap, plot_trajectory
#                       each method accepts output_dir=None; saves figure to disk when provided, shows inline otherwise
#   main.py          -- CLI entry point: argparse, constructs ExperimentConfigs, runs suite, saves all outputs
#   Add __init__.py to src/ and tests/

# TODO: Create tests/conftest.py with shared fixtures
#   Fixtures: constant_schedule, linear_schedule, exp_schedule,
#             sarsa_agent, qlearning_agent, env_manager

# TODO: Create one test file per module in tests/
#   test_schedules.py, test_agents.py, test_environment.py,
#   test_runner.py, test_experiment.py, test_visualization.py

# TODO: Sanity check CliffWalking-v0 before writing any implementation
#   Confirm: gymnasium installs cleanly, env.observation_space.n == 48, env.action_space.n == 4
#   Confirm: env.reset() returns integer state 36 (bottom-left start position)
#   Confirm: env.step() signature returns (int, float, bool, bool, dict)
#   Confirm: normal step reward == -1.0, cliff step reward == -100.0

# TODO: Verify all stubs import cleanly and pytest collects all test files
#   Run: pytest ../tests/ --collect-only

---
## Phase 2: Epsilon Schedules

In [ ]:
# TODO: Implement ConstantSchedule
#   value property always returns epsilon_start; step() is a no-op
#   Tests: value is constant across multiple step() calls

# TODO: Implement LinearDecaySchedule
#   value decays linearly from epsilon_start to epsilon_end over n_episodes steps
#   Tests: value at step 0, value at step n_episodes, value does not go below epsilon_end

# TODO: Implement ExponentialDecaySchedule
#   value = epsilon_start * (decay_rate ** episode)
#   Tests: value decreases monotonically, never reaches zero

---
## Phase 3: Agents

In [ ]:
# TODO: Implement TDAgent base class
#   Q-table initialized to zeros, shape (n_states, n_actions), dtype float
#   Tests: Q-table shape correct, dtype is float

# TODO: Implement TDAgent.select_action (epsilon-greedy)
#   Greedy: np.argmax(Q[state]); Explore: np.random.choice(n_actions)
#   Lab requires np.argmax() for greedy and np.random.choice() for exploration -- use these explicitly
#   Tests: always returns valid action in [0, n_actions); explores when epsilon=1.0; greedy when epsilon=0.0

# TODO: Implement TDAgent.reset_qtable
#   Zeros out Q-table in place
#   Tests: Q-table is all zeros after reset regardless of prior updates

# TODO: Implement TDAgent.decay_epsilon
#   Delegates to schedule.step()
#   Tests: agent epsilon reflects schedule value after decay call

# TODO: Implement SARSAAgent.update(s, a, r, s_next, a_next, terminated)
#   TD target: r + gamma * Q[s_next, a_next] (0 if terminated)
#   Q[s, a] += alpha * (target - Q[s, a])
#   Tests: Q-value moves toward target; terminal state produces zero bootstrap

# TODO: Implement QLearningAgent.update(s, a, r, s_next, a_next, terminated)
#   TD target: r + gamma * np.max(Q[s_next]) (0 if terminated); a_next is ignored
#   Q[s, a] += alpha * (target - Q[s, a])
#   Lab requires np.max() for the Q-learning target -- use explicitly
#   Tests: Q-value moves toward target; uses np.max not a_next; terminal state produces zero bootstrap

---
## Phase 4: Environment Manager

In [ ]:
# TODO: Implement EnvironmentManager
#   Wraps CliffWalking-v0; exposes n_states, n_actions
#   Tests: n_states == 48, n_actions == 4

# TODO: Implement EnvironmentManager.reset
#   Accepts optional seed; returns integer state observation
#   Tests: returns valid state in [0, n_states); CliffWalking always starts at state 36

# TODO: Implement EnvironmentManager.step
#   Returns (state, reward, terminated, truncated)
#   Tests: reward is -1 for normal step; state is valid integer; terminated and truncated are bool

---
## Phase 5: Episode Runner

In [ ]:
# TODO: Implement EpisodeRunner.run_episode
#   Full SARSA-compatible loop: select a, step, select a', update, decay epsilon at end
#   Both agents use same loop; Q-learning ignores a_next internally
#   Tests: returns a scalar total reward; reward <= -1 for any valid episode

# TODO: Implement EpisodeRunner.run_experiment
#   Outer loop over n_seeds: reset Q-table, reseed env, run n_episodes
#   Returns reward matrix of shape (n_seeds, n_episodes)
#   Tests: output shape is correct; all rewards are negative

---
## Phase 6: Experiment Framework

In [ ]:
# TODO: Implement ExperimentConfig dataclass
#   Fields: label, agent_class, alpha, epsilon_schedule, n_seeds, n_episodes
#   Tests: instantiates cleanly; fields are accessible

# TODO: Implement ExperimentResult
#   Holds config reference and reward matrix
#   Defines metric methods: final_performance, learning_speed, sample_efficiency
#   Revisit metric definitions before implementing (deferred from design discussion)
#   save(output_dir): writes reward matrix as .npy file named by config.label
#   Tests: metrics return scalar values; final_performance <= -1; save() writes a file to disk

# TODO: Implement ExperimentSuite
#   Accepts list of ExperimentConfig; runs each via EpisodeRunner
#   summarize(output_path=None): returns DataFrame; writes CSV when output_path is provided
#   Tests: DataFrame has expected columns; one row per config; CSV written when output_path given

---
## Phase 7: Baseline Experiments

In [ ]:
# TODO: Run baseline experiment
#   SARSA vs Q-learning, default params: alpha=0.1, epsilon=0.1 (constant), 30 seeds, 500 episodes
#   Confirm reward matrices are shaped (30, 500) and values are in expected range

# TODO: Plot learning curves with 95% CI
#   Y-axis: mean episode return (sum of rewards within episode) across seeds -- not per-step reward
#   CI bands: mean +/- 1.96 * np.std(matrix, axis=0) / np.sqrt(n_seeds)  -- NumPy only, no scipy
#   Both algorithms on same axes for direct comparison
#   Expected: Q-learning shows lower mean episode return during training (more cliff falls due to exploration)

---
## Phase 8: Behavioral Analysis and Visualization

In [ ]:
# TODO: Visualize learned policy as arrows on the grid
#   Greedy action at each state shown as directional arrow
#   Side-by-side: SARSA policy (safe upper path) vs Q-learning policy (cliff-hugging)

# TODO: Visualize value function heatmaps
#   max Q-value per state shown as color intensity
#   Side-by-side: SARSA vs Q-learning value surfaces

# TODO: Plot sample trajectories
#   Roll out greedy policy from trained agent (epsilon=0)
#   Show path on grid overlay for SARSA and Q-learning

# TODO: Document behavioral differences
#   Explain safe path vs cliff-hugging path in terms of on-policy vs off-policy TD targets
#   Connect to Sutton & Barto Ch. 6 discussion of SARSA vs Q-learning on CliffWalking

---
## Phase 9: Hyperparameter Experiments

In [ ]:
# TODO: Experiment with step-size alpha
#   Compare alpha in [0.05, 0.1, 0.3, 0.5] for both agents
#   Observe effect on learning speed and stability

# TODO: Experiment with epsilon schedules
#   Compare: ConstantSchedule(0.1), LinearDecaySchedule, ExponentialDecaySchedule
#   Observe: does decay cause Q-learning to eventually prefer the cliff-hugging path more reliably?
#   Observe: does decay cause SARSA to shift from safe path toward optimal path as epsilon shrinks?

---
## Phase 10: Comparison Table and Final Analysis

In [ ]:
# TODO: Generate comparison table via ExperimentSuite.summarize()
#   Columns: algorithm, alpha, epsilon_schedule, final_performance, learning_speed, sample_efficiency
#   Include all experimental configurations from Phases 7 and 9

# TODO: Write final analysis
#   Synthesize: which algorithm is better and under what conditions?
#   Address: training performance vs greedy (test-time) performance distinction
#   Address: how hyperparameters shift the safe/risky path trade-off
#   Cite: Sutton & Barto sections as appropriate

---
## Phase 11: Transfer to Submission Notebook

In [ ]:
# TODO: Populate Rudman_Craig_Lab4.ipynb -- Implementation Summary section
#   100-150 word prose: algorithms, experimental setup, key hyperparameters chosen

# TODO: Populate Rudman_Craig_Lab4.ipynb -- Key Results and Analysis section
#   Select 2-4 most informative visualizations from Phases 7-9
#   Write interpretive captions connecting results to Sutton & Barto theory
#   400-600 word analysis: behavioral differences, hyperparameter effects, what didn't work

# TODO: Populate Rudman_Craig_Lab4.ipynb -- Comparison table
#   Paste or reproduce ExperimentSuite.summarize() output

# TODO: Populate Rudman_Craig_Lab4.ipynb -- Speaker Notes (Phase 4 of lab)
#   5-7 bullet points: problem, method, design choice, key result, insight

# TODO: Populate Rudman_Craig_Lab4.ipynb -- AI Use Reflection (Section 3)
#   Document at least 2-3 concrete debugging/iteration cycles from this development process

---
## Phase 12: CLI Entry Point

In [ ]:
# TODO: Implement main.py
#   argparse arguments: --n-seeds (default 30), --n-episodes (default 500), --output-dir (default ./results)
#   Constructs all ExperimentConfigs (baseline + alpha sweep + schedule sweep)
#   Runs ExperimentSuite, calls result.save(output_dir) for each result
#   Calls Visualizer methods with output_dir to save all figures to disk
#   Calls suite.summarize(output_path) to write comparison table as CSV
#   Prints progress to stdout so long runs are observable

# TODO: Verify CLI invocation end-to-end
#   Run: python ../src/main.py --n-seeds 2 --n-episodes 10 --output-dir /tmp/test_run
#   Confirm: results directory populated with .npy files, plots, and summary.csv
#   Confirm: full production run works: python ../src/main.py --output-dir ./results